## Extracting Invasion Metrics From Simulation Outputs

In [1]:
import os
import pandas as pd
import glob
import re
from collections import defaultdict


base_path = "../../Results/output"
output_dir = "../../Results/Data/Invasion_Metrics_By_MCS"
file_pattern = "scan_*/Metrics_Data_*.csv"
os.makedirs(output_dir, exist_ok=True)

# Collect all matching file paths
file_paths = glob.glob(os.path.join(base_path, file_pattern), recursive=True)

# Containers for results
mcs_data = defaultdict(list)
combined_data_all_mcs = [] 

# Desired column order for consistency
desired_order = [
    'MCS',
    'Adhesion Energy', 'Migration Coefficient', 'Proliferative Probability',
    'Invasive Area', 'Infiltrative Area', 'Single Defects',
    'Fingers', 'Detached Cells', 'Clusters'
]

# Loop through each file
for file_path in file_paths:
    filename = os.path.basename(file_path)
    
    # Extract parameters from filename
    match = re.search(r'Metrics_Data_(-?[\d.]+)_([\d.]+)_([\d.]+)', filename)
    if not match:
        print(f"Skipping {filename}: pattern mismatch.")
        continue

    try:
        contact_energy = float(match.group(1).rstrip('.'))
        chemotaxis_lambda = float(match.group(2).rstrip('.'))
        proliferative_probability = float(match.group(3).rstrip('.'))
    except ValueError as e:
        print(f"Skipping {filename}: parameter conversion error – {e}")
        continue

    # Load file
    try:
        df = pd.read_csv(file_path, sep=",")
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        continue

    # Validate columns
    if 'MCS' not in df.columns:
        print(f"Skipping {filename}: 'MCS' column not found.")
        continue

    # Filter for per-MCS outputs (every 100 MCS)
    df_filtered_100 = df[df['MCS'] % 100 == 0].copy()
    # Filter for combined output (every 10 MCS)
    df_filtered_10 = df[df['MCS'] % 10 == 0].copy()

    # Add parameter annotations
    for filtered_df in [df_filtered_100, df_filtered_10]:
        if not filtered_df.empty:
            filtered_df['Adhesion Energy'] = contact_energy
            filtered_df['Migration Coefficient'] = chemotaxis_lambda
            filtered_df['Proliferative Probability'] = proliferative_probability

    # Process per-MCS files
    for mcs_value, group_df in df_filtered_100.groupby('MCS'):
        group_df = group_df.copy()
        group_df['MCS'] = int(mcs_value)

        # Reorder columns for consistency
        group_df = group_df[[col for col in desired_order if col in group_df.columns]]

        mcs_data[mcs_value].append(group_df)

    # Accumulate for combined output (row-wise)
    if not df_filtered_10.empty:
        combined_data_all_mcs.append(df_filtered_10)

# Write individual files per MCS
for mcs, dfs in mcs_data.items():
    full_df = pd.concat(dfs, ignore_index=True)
    full_df = full_df.sort_values(by=['Adhesion Energy', 'Migration Coefficient', 'Proliferative Probability'])

    out_file = os.path.join(output_dir, f"invasion_metrics_{mcs}_mcs.csv")
    full_df.to_csv(out_file, index=False)

# Write combined file (across all MCS values filtered by MCS % 10)
if combined_data_all_mcs:
    combined_df = pd.concat(combined_data_all_mcs, ignore_index=True)
    combined_df['MCS'] = combined_df['MCS'].astype(int)
    combined_df = combined_df.sort_values(by=['MCS', 'Adhesion Energy', 'Migration Coefficient', 'Proliferative Probability'])

    # Reorder columns for consistency
    combined_df = combined_df[[col for col in desired_order if col in combined_df.columns]]

    combined_out_file = os.path.join(output_dir, "combined_invasion_metrics.csv")
    combined_df.to_csv(combined_out_file, index=False)
    print(f"📁 Combined CSV saved: {combined_out_file}")

print(f"✅ Done. Sorted metrics saved by MCS and as one combined file to: {output_dir}")


📁 Combined CSV saved: ../../Results/Data/Invasion_Metrics_By_MCS/combined_invasion_metrics.csv
✅ Done. Sorted metrics saved by MCS and as one combined file to: ../../Results/Data/Invasion_Metrics_By_MCS


## PHENOTYPE CLASSIFICATION

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the CSV file
file_path =  "../../Results/Data/invasion_metrics.csv"
df = pd.read_csv(file_path)

# Define the phenotype classification function
def classify_phenotype(row):
    invasive_area = row["Invasive Area"]
    infiltrative_area = row["Infiltrative Area"]
    branches = row["Fingers"]
    single_defect = row["Single Defects"]
    clusters = row["Clusters"]
    
    
    if invasive_area == infiltrative_area and branches == 0  and single_defect == 0 and clusters == 0:
        return "No invasion"
    elif invasive_area < infiltrative_area and branches == 0  and single_defect > 0 and clusters == 0:
        return "Single cell invasion"
    elif invasive_area == infiltrative_area and branches > 0 and single_defect == 0 and clusters == 0:
        return "Bulk invasion"
    elif invasive_area < infiltrative_area and branches > 0 and single_defect > 0 and clusters >= 0:
        return "Multimodal invasion"
    else:
        return "Unclassified"

df["Phenotype"] = df.apply(classify_phenotype, axis=1)

df = df[df["Phenotype"] != "Unclassified"].reset_index(drop=True)

output_file_path = "../../Results/Data/phenotype_classification.csv"

df.to_csv(output_file_path, index=False)

print(f"Phenotype classification saved to: {output_file_path}")


def classify_phenotype(row):
    branches = row["Fingers"]
    single_defect = row["Single Defects"]
    clusters = row["Clusters"]
    
    if (branches == 0 or branches == 1) and single_defect == 0 and clusters == 0:
        return "No invasion"
    elif (branches == 0 or branches == 1) and single_defect > 0 and clusters == 0:
        return "Single cell invasion"
    elif branches > 0 and single_defect == 0 and clusters == 0:
        return "Bulk invasion"
    elif branches > 0 and single_defect >= 0 and clusters >= 0:
        return "Multimodal invasion"
    else:
        return "Unclassified"

df["Phenotype"] = df.apply(classify_phenotype, axis=1)


param_cols = ["Contact Energy", "Chemotaxis Lambda", "Proliferative Probability", "Phenotype"]
df_filtered = df[param_cols]

phenotype_counts = df_filtered.groupby(["Contact Energy", "Chemotaxis Lambda", "Proliferative Probability", "Phenotype"]).size().reset_index(name="Count")

# Normalize to get probability
phenotype_probs = phenotype_counts.copy()
phenotype_probs["Total"] = phenotype_probs.groupby(["Contact Energy", "Chemotaxis Lambda", "Proliferative Probability"])["Count"].transform("sum")
phenotype_probs["Probability"] = phenotype_probs["Count"] / phenotype_probs["Total"]

phenotype_probs = phenotype_probs[["Contact Energy", "Chemotaxis Lambda", "Proliferative Probability", "Phenotype", "Probability"]]


output_file_path = "../../Results/Data/phenotype_probability_map.csv"
phenotype_probs.to_csv(output_file_path, index=False)

print(f"Phenotype probability map saved to: {output_file_path}")


Phenotype classification saved to: ../../Results/Data/phenotype_classification.csv
Phenotype probability map saved to: ../../Results/Data/phenotype_probability_map.csv
